# TASVEG - The Digital Vegetation Map of Tasmania

> TASVEG is a comprehensive digital map of Tasmania's vegetation, including sub-Antarctic Macquarie Island. The map depicts the extent of more than 150 vegetation communities, including coastal heathlands, eucalypt forest and alpine communities. 

The technical manual includes description for each of the 162 mapping units, most of them ecological vegetation communities. The format of these descriptions is consistent, descriptive and cross-referenced with relevant literature and other processes. 


## Links 

- Technical manual with full description of communities: [From Forest to Fjaeldmark: Descriptions of Tasmania's Vegetation (Edition 2).](https://dpipwe.tas.gov.au/conservation/flora-of-tasmania/from-forest-to-fjaeldmark-descriptions-of-tasmanias-vegetation)
- Data download from Tasmania's open data reporitory [`theLIST`](https://listdata.thelist.tas.gov.au/opendata/)
- [Technical Notes and Metadata documentation](https://nre.tas.gov.au/conservation/development-planning-conservation-assessment/planning-tools/monitoring-and-mapping-tasmanias-vegetation-(tasveg)/tasveg-the-digital-vegetation-map-of-tasmania/tasveg-documentation)

## Version and license information

> The current release version for mainland Tasmania is TASVEG 5.0 and was published in November 2025. The current release version for Macquarie Island is TASVEG 3.0 Macquarie Island and was published in November 2013.

Data is supplied under Creative Commons Attribution 3.0 Australia (CC BY 3.0 AU) Licence http://creativecommons.org/licenses/by/3.0/au/ 

The Department Natural Resources and Environment Tasmania requests attribution be provided in the following manner: “TASVEG 5.0 from www.theLIST.tas.gov.au ©State of Tasmania” 

## Citation

**TASVEG 5.0**
> Department of Natural Resources and Environment Tasmania. TASVEG 5.0, Released November 2025. Tasmanian Vegetation Monitoring and Mapping Program, Environment Business Unit.​

​
**TASVEG 3.0 Macquarie Island**
> Department of Natural Resources and Environment Tasmania. TASVEG 3.0 Macquarie Island, Released November 2013. Tasmanian Vegetation Monitoring and Mapping Program, Resource Management and Conservation Division.​

**Technical manual**

Kitchener, A. & Harris, S. (2013). From Forest to Fjaeldmark​: Descriptions of Tasmania's Vegetation. Edition 2. Department of Primary Industries, Parks, Water and Environment, Tasmania.
Available at http://nre.tas.gov.au/conservation/flora-of-tasmania/from-forest-to-fjaeldmark-descriptions-of-tasmanias-vegetation​ [Verified 4 Jul 2026].

## Data Download

### Set-up

In [2]:
import pyprojroot
import os
from pathlib import Path
import sys
import json
import requests
# for reading xml files (metadata)
#from xml.etree import ElementTree
import xmltodict

In [3]:
import zipfile, requests, io

In [4]:
repodir = pyprojroot.find_root(pyprojroot.has_dir(".git"))

### Metadata

In [14]:
portal_url = 'https://data.thelist.tas.gov.au/datagn/srv/eng/main.home'
params = {'uuid': 'b5c7a079-14bc-4b3c-af73-db7585d34cdd'}

In [15]:
portal_url = 'https://www.thelist.tas.gov.au/app/content/data/download'
params = {'downloadType': 'xml',
          'detailUID': 'b5c7a079-14bc-4b3c-af73-db7585d34cdd'}

In [16]:
r = requests.get(portal_url, params=params)

print("Collection metadata URL: {0}\n".format(r.url))

Collection metadata URL: https://www.thelist.tas.gov.au/app/content/data/download?downloadType=xml&detailUID=b5c7a079-14bc-4b3c-af73-db7585d34cdd



In [17]:
print(r.status_code,r.reason)

200 200


In [18]:
dict_data = xmltodict.parse(r.content)
metadata = dict_data['csw:GetRecordByIdResponse']['gmd:MD_Metadata']

In [10]:
for key in metadata.keys():
    print(key)

@xmlns:gmd
@xmlns:gco
@xmlns:gmx
@xmlns:srv
@xmlns:gts
@xmlns:gml
@xmlns:xlink
@xmlns:geonet
gmd:fileIdentifier
gmd:hierarchyLevel
gmd:contact
gmd:dateStamp
gmd:referenceSystemInfo
gmd:identificationInfo
gmd:distributionInfo
gmd:dataQualityInfo
gmd:metadataConstraints


In [20]:
metadata['gmd:dataQualityInfo']['gmd:DQ_DataQuality']['gmd:lineage']['gmd:LI_Lineage']

{'gmd:statement': {'gco:CharacterString': "The first version of TASVEG (TASVEG 1.0) was released in 2004. The initial TASVEG release incorporated existing information from the Regional Forest Agreement mapping program and from the World Heritage Area (WHA) mapping program, with remaining areas of the state mapped by the TVMMP. Since TASVEG 1.0 there have been three minor versions released; 1.1 and 1.2 released in 2005 and 1.3 released in 2007. Two major versions have been released since TASVEG 1.0, these were 2.0 in 2009 and 3.0 in 2013. Each new version of TASVEG has been accompanied by a statement indicating which new mapping data has been incorporated since the last release.\n\nPhotographic interpretation (PI) of DPIPWE's most current aerial photography is the primary method of data collection for TASVEG updates, with field verification of representative polygons undertaken where practicable. Some aerial photography is analysed stereoscopically where feasible, but the majority of im

In [22]:
metadata['gmd:distributionInfo']

{'gmd:MD_Distribution': {'gmd:distributionFormat': [{'gmd:MD_Format': {'gmd:name': {'gco:CharacterString': 'ESRI_shp'},
     'gmd:version': None}},
   {'gmd:MD_Format': {'gmd:name': {'gco:CharacterString': 'Other'},
     'gmd:version': {'gco:CharacterString': 'ESRI file geodatabase'}}}],
  'gmd:transferOptions': [{'gmd:MD_DigitalTransferOptions': {'gmd:onLine': {'gmd:CI_OnlineResource': {'gmd:linkage': {'gmd:URL': None},
       'gmd:name': {'gco:CharacterString': 'View Dataset'}}}}},
   {'gmd:MD_DigitalTransferOptions': {'gmd:onLine': {'gmd:CI_OnlineResource': {'gmd:linkage': {'gmd:URL': None},
       'gmd:name': {'gco:CharacterString': 'Download Data'}}}}},
   {'gmd:MD_DigitalTransferOptions': {'gmd:onLine': {'gmd:CI_OnlineResource': {'gmd:linkage': {'gmd:URL': 'https://dpipwe.tas.gov.au/conservation/development-planning-conservation-assessment/planning-tools/monitoring-and-mapping-tasmanias-vegetation-(tasveg)/tasveg-the-digital-vegetation-map-of-tasmania/tasveg-documentation'},
    

### Data download

The metadata above has an empty value for the download url, but the data is available at:
https://listdata.thelist.tas.gov.au/opendata/
We just need to reverse engineer the link from the download links:

In [23]:
data_url = 'https://listdata.thelist.tas.gov.au/opendata/data/LIST_TASVEG_50_{}.zip'
# zip_url = data_url.format('FIRE_ATTRIBUTES_STATEWIDE')
zip_url = data_url.format('STATEWIDE')

In [32]:
output_dir = repodir / 'gisdata' / 'TAS' / 'TASVEG_5.0'
# Path(output_dir).parent.mkdir(parents=True, exist_ok=True)

In [34]:
# Extract only if the target directory does not exist
if not os.path.exists(output_dir):
    r = requests.get(zip_url)
    with zipfile.ZipFile(io.BytesIO(r.content)) as zip_ref:
        zip_ref.extractall(output_dir)
    print("Extraction complete.")
else:
    print("Directory already exists. Extraction skipped.")

Extraction complete.


Test the downloaded file:

In [35]:
import geopandas as gpd

In [36]:
tasveg = gpd.read_file(output_dir / 'LIST_TASVEG_50_STATEWIDE/TASVEG_5_0.shp')